# Entity resolver walkthrough (`lib/entity_resolver.py`)

This notebook explains **how literature entities map to PrimeKG-Plus nodes**.

The source of truth is `lib/entity_resolver.py` — this notebook **imports that module** and does not duplicate logic.

## Resolution order (same as FINAL-* mapping notebooks)

1. **Name match** — `entity1` / `entity1_suggested_name` / `second_search_suggested_name` vs `nodes.csv`
2. **CUI → ontology id** — UMLS CUI (valid when assigned by curators) → id by entity type:
   - disease → MONDO · phenotype/pathology → HPO · gene → NCBI (fallback HGNC symbol)
   - drug → DrugBank · GO types → GO · anatomy → UBERON · pathway → REACTOME
   - xref: `hp_references` / `mondo_references` + **UMLS atoms** (`umls_2025AB.csv`)
3. **Create node** — if the ontology id is valid but absent from the KG → add a node (label from ontology terms)
4. **Loose name** — `entity_status == in_kg` but the name does not match → try the suggested name

## Notes

- HPO / GO ids in `nodes.csv` **omit leading zeros** → the resolver normalizes before lookup.
- A CUI without a MONDO/HPO/NCBI/DrugBank/GO atom in UMLS may still be skipped even if curation-valid.


In [1]:
from pathlib import Path
import os
import sys

import pandas as pd

SCRIPT_DIR = Path.cwd()
if not (SCRIPT_DIR / "lib" / "entity_resolver.py").exists():
    SCRIPT_DIR = Path("/Users/ljw303/YANG_DATA/PrimeKG/PrimeKG-Plus_release/scripts/literature_curation")

sys.path.insert(0, str(SCRIPT_DIR))
from lib.entity_resolver import (
    EntityResolver,
    parse_cuis,
    normalize_node_id,
    norm_name,
)

RELEASE_ROOT = SCRIPT_DIR.parents[1]
PRIMEKG_ROOT = Path(os.environ.get("PRIMEKG_ROOT", "/Users/ljw303/YANG_DATA/PrimeKG"))
DATA_DIR = PRIMEKG_ROOT / "datasets" / "data"
NODES_PATH = Path(os.environ.get("PLUS_NODES", RELEASE_ROOT / "dataset" / "nodes.csv"))

nodes = pd.read_csv(NODES_PATH, low_memory=False)
print(f"nodes.csv: {len(nodes):,} rows")
resolver = EntityResolver(nodes, DATA_DIR)
print(f"CUI→MONDO map: {len(resolver._cui_to_mondo):,}")
print(f"CUI→HPO map:   {len(resolver._cui_to_hp):,}")


nodes.csv: 129,317 rows
CUI→MONDO map: 21,480
CUI→HPO map:   12,762


In [2]:
NODES_PATH

PosixPath('/Users/ljw303/YANG_DATA/PrimeKG/PrimeKG-Plus_release/dataset/nodes.csv')

### Demo 1 — HPO leading-zero bug (fixed)

`C0544820` → HPO `0003429` = **CNS hypomyelination**; in `nodes.csv` the id is `3429`.

In [3]:
cui = "C0544820"
oid_raw = resolver._cui_to_hp.get(cui)
oid_norm = normalize_node_id(oid_raw, "phenotype")
print("CUI:", cui)
print("HPO ref id:", oid_raw, "→ normalized:", oid_norm)
print("In nodes.csv:", len(nodes[(nodes.node_id.astype(str) == oid_norm) & (nodes.node_source == "HPO")]))

hit_old_style = resolver._by_key.get((oid_raw, "effect/phenotype", "HPO"))
hit_new = resolver.resolve_or_create("hypomyelination", cui, "phenotype")
print("Lookup with raw oid:", hit_old_style)
print("resolve_or_create:", hit_new)


CUI: C0544820
HPO ref id: 0003429 → normalized: 3429
In nodes.csv: 1
Lookup with raw oid: None
resolve_or_create: ResolvedNode(node_id='3429', node_type='effect/phenotype', node_name='CNS hypomyelination', node_source='HPO', method='cui')


### Demo 2 — Disease CUI: node not yet in KG → create new

`C0085078` → MONDO `2561` (lysosomal storage disease) — may not yet exist in `primekg_plus.csv`.

In [4]:
cui = "C0085078"
resolver2 = EntityResolver(nodes, DATA_DIR)  # fresh resolver
hit = resolver2.resolve_or_create("lysosomal storage disorder", cui, "disease")
print("Resolved:", hit)
print("New nodes created:", len(resolver2.new_nodes))
if resolver2.new_nodes:
    display(pd.DataFrame(resolver2.new_nodes))


Resolved: ResolvedNode(node_id='2561', node_type='disease', node_name='lysosomal storage disease', node_source='MONDO', method='new_cui')
New nodes created: 1


,node_id,node_type,node_name,node_source
0,2561,disease,lysosomal storage disease,MONDO


### Demo 3 — Resolve real entities from the curation pool

In [5]:
from integrate_primekg_plus_rd import collect_literature_rows, default_disease_specs

CURATION_ROOT = Path(os.environ.get("CURATION_ROOT", "/Users/ljw303/YANG_DATA/THUY_DATA_CURATION"))
literature = collect_literature_rows(default_disease_specs(CURATION_ROOT))
print("Curated pool:", len(literature), "rows")

sample = literature.head(8)
rows = []
for _, r in sample.iterrows():
    for side, (name, status, etype, sug, expert) in {
        1: (r.entity1, r.entity1_status, r.entity_type1, r.get("entity1_suggested_name"), r.get("entity1_expert_cui")),
        2: (r.entity2, r.entity2_status, r.entity_type2, r.get("entity2_suggested_name"), r.get("entity2_expert_cui")),
    }.items():
        hit = resolver.resolve_or_create(name, status, etype, sug, r.get("second_search_suggested_name"), expert)
        rows.append({
            "entity": name, "type": etype, "status": status,
            "expert_cui": expert, "method": hit.method if hit else None,
            "node": f"{hit.node_source}:{hit.node_id}" if hit else None,
        })
pd.DataFrame(rows)


Curated pool: 1290 rows


,entity,type,status,expert_cui,method,node
0,Canavan disease,disease,in_kg,None,name,MONDO_grouped:10079_16781_16207
1,leukodystrophy,disease,in_kg,None,name,MONDO:19046
2,Canavan disease,disease,in_kg,None,name,MONDO_grouped:10079_16781_16207
3,macrocephaly,phenotype,in_kg,None,name,HPO:256
4,Canavan disease,disease,in_kg,None,name,MONDO_grouped:10079_16781_16207
5,neurologic impairment,phenotype,C1854838|C5233653,None,suggested_name,HPO:2344
6,Canavan disease,disease,in_kg,None,name,MONDO_grouped:10079_16781_16207
7,ASPA,protein/gene,in_kg,None,name,NCBI:443
8,Canavan disease,disease,in_kg,None,name,MONDO_grouped:10079_16781_16207
9,ASPA,protein/gene,in_kg,None,name,NCBI:443


### Demo 4 — Entity types and current CUI bridges

The resolver **only** maps CUI→node for `disease` and `phenotype`. Other types require a direct name match in the KG.

In [6]:
from lib.entity_resolver import ENTITY_TYPE_TO_NODE, CREATE_NODE_SOURCE
pd.DataFrame([
    {"entity_type": k, "node_type": v[0], "sources": ", ".join(sorted(v[1])), "cui_create": k in CREATE_NODE_SOURCE}
    for k, v in ENTITY_TYPE_TO_NODE.items()
]).drop_duplicates(subset=["entity_type"]).sort_values("entity_type")


,entity_type,node_type,sources,cui_create
9,anatomy,anatomy,UBERON,False
4,biological_process,biological process,GO,False
5,cellular_component,cellular component,GO,False
0,disease,disease,"MONDO, MONDO_grouped",True
8,drug,drug,DrugBank,False
11,exposure,exposure,CTD,False
3,gene/protein,gene/protein,NCBI,False
6,molecular function,molecular function,GO,False
7,molecular_function,molecular function,GO,False
10,pathway,pathway,REACTOME,False
